# `wf_account_list` 단계별 Testbed

전체 계좌 목록 조회에서 Agent 내부 추출, Backend Tool 요청과 결과 Webhook을 순서대로 확인한다.

## 전체 흐름

```text
사용자 발화 → 계좌 힌트 추출 → GET /agent-tools/accounts → account_list Webhook → 종료
```

Agent는 계좌번호 원문과 잔액을 조회하거나 결과에 포함하지 않는다.

In [ ]:
import json

from pydantic import SecretStr

from agent.clients.backend import BackendClientConfig
from agent.testing.account_list import create_account_list_mock_testbed
from agent.testing.mock_backend import MockBackend
from agent.workflow_contracts import WorkflowContractStore
from agent.workflows.account_list import extract_account_list_slots_from_text
from agent.workflows.query_slot_extraction import extract_account_list_slots_llm_first


def show(title, value):
    print(f"\n--- {title} ---")
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


config = BackendClientConfig(
    base_url="http://backend.test",
    agent_service_token=SecretStr("test-token"),
    agent_webhook_secret=SecretStr("test-secret"),
    retry_backoff_seconds=0,
)
print("설정 완료")

## 1. 계약과 Agent 내부 LLM 우선 추출 확인

구조화 LLM 추출을 먼저 시도하고, 모델을 사용할 수 없거나 결과가 유효하지 않을 때만 결정적 규칙으로 폴백한다.

In [ ]:
workflow = WorkflowContractStore().get_workflow("wf_account_list")
show("Step 계약", [
    {"step_id": step["step_id"], "mode": step["interaction_mode"], "contract_id": step.get("contract_id")}
    for step in workflow["steps"]
])
message = "생활비 계좌 찾아줘"
show("LLM 우선 추출 결과", await extract_account_list_slots_llm_first(message))
show("결정적 폴백 참고", extract_account_list_slots_from_text(message))

## 2. Mock Backend 응답 준비

In [ ]:
backend = MockBackend()
backend.add_success("GET", "/api/v1/agent-tools/accounts", {"accounts": [{
    "account_id": "acc_001",
    "bank_name": "토스뱅크",
    "account_alias": "생활비 계좌",
    "account_type": "checking",
    "masked_account_number": "1000-***-1234",
    "currency": "KRW",
    "is_default": True,
    "status": "active",
}]})
backend.add_success("POST", "/api/v1/webhooks/agent", {"message_id": "message_001"})
print("Mock 응답 준비 완료")

## 3. Workflow 실행과 결과 확인

In [ ]:
testbed = create_account_list_mock_testbed(backend, config, thread_id="thread_account_list")
result = await testbed.start(
    message="생활비 계좌 찾아줘",
    request_id="req_account_list",
    chat_session_id="chat_001",
    execution_context_id="exec_001",
)
show("실행 결과", result.model_dump(mode="json"))
show("HTTP 요청·응답", backend.exchange_timeline(include_payload=True))
show("최종 State", await testbed.state("thread_account_list"))
await testbed.aclose()